Trouble shooting stochastic part of the code

In [12]:
#FEA PART
using LinearAlgebra, Printf,SparseArrays,StaticArrays
using Plots,Dates,Statistics, DelimitedFiles
using Ferrite, FerriteMeshParser
using TickTock, Parameters, Random
using IterativeSolvers
using WriteVTK

using Ferrite
using LinearAlgebra
using Random
using Distributions
using Arpack


In [13]:

include("input.jl")
include("opt.jl")
include("FE_updated_stoch.jl")


mp = MaterialParams(λ, μ_l, μ_t, alpha, beta, angle)

corners = [
    Vec{2}((0.0, 0.0)),
    Vec{2}((lx, 0.0)),
    Vec{2}((lx, ly)),
    Vec{2}((0.0, ly)),
]

grid = generate_grid(Quadrilateral, (nelx, nely), corners)

addnodeset!(grid, "left_edge", x -> norm(x[1]) ≈ 0.0 );
addnodeset!(grid, "right_edge", x -> norm(x[1]) ≈ lx );
addfaceset!(grid, "right_face", x -> x[1] ≈ lx && norm(x[2] - ly) <= 0.5)

dim = 2;
ip_g = Lagrange{dim, RefCube, 1}();
ip = Lagrange{dim, RefCube, 1}(); # other option would be 8 noded Serendipity elements
qpo = 2;
qr = QuadratureRule{dim, RefCube}(qpo);
cv = CellScalarValues(Float64,qr, ip, ip_g);

fqr = QuadratureRule{dim-1,RefCube}(qpo);
fv = FaceVectorValues(fqr, ip, ip_g);

# one gauss point integration for postprocessing
nqp_post = 1;
qr_post = QuadratureRule{dim, RefCube}(nqp_post);
cv_post = CellScalarValues(Float64,qr_post, ip, ip_g);

#ΓN = getfaceset(grid, "right_face"); # Neumann Boundary

dh = DofHandler(grid);
push!(dh, :u, 2, ip); # Displacement vector
close!(dh);

ch = ConstraintHandler(dh);

∂Ωl = getnodeset(dh.grid, "left_edge");
dbclh = Ferrite.Dirichlet(:u, ∂Ωl, (x,t) -> 1e-10, 1); # fix the left edge
add!(ch, dbclh);
dbclv = Ferrite.Dirichlet(:u, ∂Ωl, (x,t) -> 1e-10, 2); # fix the left edge
add!(ch, dbclv);
"""∂Ωr = getnodeset(dh.grid, "right_edge");
dbcrv = Dirichlet(:u, ∂Ωr, (x,t) -> 0.2*t, 2); # Vertical Displacement
add!(ch, dbcrv);"""

close!(ch);
update!(ch, 0.0);


# dof vector
uₙ = Vector{Float64}(undef,ndofs(dh));
fill!(uₙ,zero(eltype(uₙ)));

u = Vector{Float64}(undef,ndofs(dh));
fill!(u,zero(eltype(u)));
u .= uₙ;

save_path = mkpath("./FEOutputs/$(basename(@__DIR__))");
#clean_savepath(save_path)
exportresults(uₙ, dh, grid, cv_post, mp, ip, save_path, 0)


cells = getcells(grid)
elements = [cell.nodes for cell in cells]
nodes = getnodes(grid)
nelem = length(elements)
nnodes = length(nodes)

nloc = ndofs_per_cell(dh)
Bmat = Array{Float64}(undef, 3, nloc)
ϵ   = Vector{Float64}(undef, 3)
σ   = similar(ϵ)
ℂ   = Array{Float64}(undef, 3, 3)


coords = getcoordinates(grid, 1)

4-element Vector{Vec{2, Float64}}:
 [0.0, 0.0]
 [0.5, 0.0]
 [0.5, 0.5]
 [0.0, 0.5]

In [14]:
dim = 2
nloc = length(cells[1].nodes)   # 4 for quadrilateral
nelem = length(cells)

# store a Vec{2,Float64} for each element/node: shape (nelem, nloc)
coords_elem = Array{Vec{2,Float64}}(undef, nelem, nloc)
for ei in 1:nelem
    v = getcoordinates(grid, ei)   # Vector{Vec{2,Float64}} of length nloc
    @assert length(v) == nloc
    for ni in 1:nloc
        coords_elem[ei, ni] = v[ni]
    end
end

coords_elem[1,:]

4-element Vector{Vec{2, Float64}}:
 [0.0, 0.0]
 [0.5, 0.0]
 [0.5, 0.5]
 [0.0, 0.5]

In [15]:
# ...existing code...

function covariance_matrix_from_elemcoords(coords_elem::AbstractArray, σ::Float64, Lc::Float64;
                                          use_centroids::Bool=false,
                                          make_sparse::Bool=false,
                                          cutoff_mult::Float64=3.0,
                                          kernel::Symbol = :exponential,
                                          matern_nu::Float64 = 1.5,
                                          eltype_out=Float64)
    nelem, nloc = size(coords_elem)

    # collect points
    if use_centroids
        pts = Vector{typeof(coords_elem[1,1])}(undef, nelem)
        for ei in 1:nelem
            s = zero(coords_elem[1,1])
            for ni in 1:nloc
                s += coords_elem[ei, ni]
            end
            pts[ei] = s / nloc
        end
    else
        pts = Vector{typeof(coords_elem[1,1])}(undef, nelem * nloc)
        k = 1
        for ei in 1:nelem
            for ni in 1:nloc
                pts[k] = coords_elem[ei, ni]
                k += 1
            end
        end
    end

    n = length(pts)
    if n == 0
        return zeros(eltype_out,0,0), pts
    end

    cutoff = cutoff_mult * Lc

    kernel_val = function(r)
        if kernel == :exponential
            return (σ^2) * exp(-r / Lc)
        elseif kernel == :gaussian
            return (σ^2) * exp(-(r^2) / (2 * Lc^2))
        elseif kernel == :matern
            # simple Matern(ν) approximate using (ν=1.5 or 2.5 common)
            if isapprox(matern_nu, 1.5; atol=1e-8)
                s = sqrt(3.0) * r / Lc
                return (σ^2) * (1.0 + s) * exp(-s)
            elseif isapprox(matern_nu, 2.5; atol=1e-8)
                s = sqrt(5.0) * r / Lc
                return (σ^2) * (1.0 + s + (s^2)/3.0) * exp(-s)
            else
                # fallback to exponential
                return (σ^2) * exp(-r / Lc)
            end
        else
            return (σ^2) * exp(-r / Lc)
        end
    end

    if make_sparse
        I = Int[]; J = Int[]; V = eltype_out[]
        for i in 1:n
            pi = pts[i]
            for j in i:n
                r = norm(pi - pts[j])
                if r <= cutoff
                    push!(I, i); push!(J, j); push!(V, eltype_out(kernel_val(r)))
                    if i != j
                        push!(I, j); push!(J, i); push!(V, eltype_out(kernel_val(r)))
                    end
                end
            end
        end
        C = sparse(I, J, V, n, n)
    else
        C = zeros(eltype_out, n, n)
        for i in 1:n
            for j in i:n
                r = norm(pts[i] - pts[j])
                val = kernel_val(r)
                C[i,j] = eltype_out(val)
                C[j,i] = C[i,j]
            end
        end
    end

    return C, pts
end

function KL_realization(material_params::MaterialParams, coords_elem::AbstractArray;
                        σs=Dict{Symbol,Float64}(), Lc=0.1, N_modes=5,
                        use_centroids=false, make_sparse=true, eltype_out=Float32,
                        kernel::Symbol = :gaussian, matern_nu::Float64=1.5,
                        mode::Symbol = :additive)   # :additive or :lognormal (multiplicative)
    nelem, nloc = size(coords_elem)
    mat_field = Dict{Symbol,Any}()

    params = [
        (:μ_l, material_params.μ_l),
        (:μ_t, material_params.μ_t),
        (:α,  material_params.alpha),
        (:β,  material_params.beta),
    ]

    for (sym, μ_bar) in params
        # default larger sigma relative to mean for stronger changes
        σ_default = 0.25 * abs(μ_bar)   # increase from 0.1 -> 0.25
        σ = haskey(σs, sym) ? σs[sym] : σ_default

        C, pts = covariance_matrix_from_elemcoords(coords_elem, σ, Lc;
                                                   use_centroids=use_centroids,
                                                   make_sparse=make_sparse,
                                                   cutoff_mult=3.0,
                                                   kernel=kernel,
                                                   matern_nu=matern_nu,
                                                   eltype_out=eltype_out)
        n = size(C,1)
        if n == 0
            mat_field[sym] = use_centroids ? fill(μ_bar, nelem) : fill(μ_bar, nelem, nloc)
            continue
        end

        n_ev = min(N_modes, n)
        if n <= 2000 && !issparse(C)
            λs, ϕs = eigen(Symmetric(Matrix(C)))
            idx = sortperm(λs, rev=true)[1:n_ev]
            λs_trunc = λs[idx]; ϕs_trunc = ϕs[:, idx]
        else
            try
                @eval using Arpack
                vals, vecs = Arpack.eigs(C; nev=n_ev, which=:LM)
                λs_trunc = real(vals)
                ϕs_trunc = real(vecs)
            catch err
                @warn "Arpack failed — falling back to dense eigen: $err"
                λs, ϕs = eigen(Symmetric(Matrix(C)))
                idx = sortperm(λs, rev=true)[1:n_ev]
                λs_trunc = λs[idx]; ϕs_trunc = ϕs[:, idx]
            end
        end

        ξ = randn(n_ev)
        if mode == :additive
            field_vals = μ_bar .+ ϕs_trunc * (sqrt.(λs_trunc) .* ξ)
        else
            # log-normal multiplicative: sample Gaussian field then exponentiate
            g = ϕs_trunc * (sqrt.(λs_trunc) .* ξ)   # zero-mean Gaussian
            field_vals = μ_bar .* exp.(g)           # multiplicative changes
        end

        if use_centroids
            mat_field[sym] = field_vals
        else
            field_elem = Array{Float64}(undef, nelem, nloc)
            k = 1
            for ei in 1:nelem, ni in 1:nloc
                field_elem[ei, ni] = float(field_vals[k])
                k += 1
            end
            mat_field[sym] = field_elem
        end
    end

    for (sym, val) in [(:λ, material_params.λ), (:angle, material_params.angle)]
        if use_centroids
            mat_field[sym] = fill(val, nelem)
        else
            mat_field[sym] = fill(val, nelem, nloc)
        end
    end

    return mat_field
end

# ...existing code...

KL_realization (generic function with 1 method)

In [16]:
mp = MaterialParams(λ, μ_l, μ_t, alpha, beta, angle)
fields = KL_realization(mp, coords_elem;
                        σs=Dict(:μ_l=>2.0, :μ_t=>0.8, :α=>1.0, :β=>0.5),
                        Lc=0.5, N_modes=40, use_centroids=false,
                        make_sparse=true, kernel=:gaussian, mode=:lognormal)

Dict{Symbol, Any} with 6 entries:
  :α     => [5.00675 5.01651 5.03791 5.01553; 5.01651 5.02929 5.06742 5.03791; …
  :λ     => [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0; … ; 1.0 1.0 1.0 1.0; 1.0 1.0 1.0…
  :angle => [0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0; … ; 0.0 0.0 0.0 0.0; 0.0 0.0 0.0…
  :μ_l   => [9.93522 9.84875 9.65306 9.84988; 9.84875 9.75024 9.43117 9.65306; …
  :μ_t   => [1.00059 1.00136 1.00331 1.00144; 1.00136 1.00219 1.00535 1.00331; …
  :β     => [1.99532 1.98902 1.97508 1.9893; 1.98902 1.98177 1.95875 1.97508; ……

In [10]:
using Printf

function pretty_print_fields(fields; n_elem_display::Int=8)
    for (k,v) in fields
        println("Field: ", k)
        if ndims(v) == 2
            n_e, n_loc = size(v)
            gmin, gmax = minimum(v), maximum(v)
            gmean, gstd = mean(v), std(v)
            ranges = [maximum(v[i,:]) - minimum(v[i,:]) for i in 1:n_e]
            rmin, rmean, rmax = minimum(ranges), mean(ranges), maximum(ranges)
            @printf("  size = %d × %d\n", n_e, n_loc)
            @printf("  global min / max / mean / std = %.6g / %.6g / %.6g / %.6g\n",
                    gmin, gmax, gmean, gstd)
            @printf("  per-element range min / mean / max = %.6g / %.6g / %.6g\n",
                    rmin, rmean, rmax)
            nshow = min(n_elem_display, n_e)
            shown = join([@sprintf("%.6g", ranges[i]) for i in 1:nshow], ", ")
            println("  first $(nshow) per-element ranges: ", shown)
        else
            @printf("  vector len = %d\n", length(v))
            @printf("  min / max / mean / std = %.6g / %.6g / %.6g / %.6g\n",
                    minimum(v), maximum(v), mean(v), std(v))
        end
        println()
    end
end

pretty_print_fields (generic function with 1 method)

In [11]:
pretty_print_fields(fields, n_elem_display=8)

Field: α
  size = 4800 × 4
  global min / max / mean / std = 2.91466 / 8.8785 / 5.06658 / 0.960256
  per-element range min / mean / max = 0.00919008 / 0.278643 / 1.00489
  first 8 per-element ranges: 0.0486778, 0.0718795, 0.0889151, 0.100571, 0.106628, 0.106596, 0.104036, 0.106748

Field: λ
  size = 4800 × 4
  global min / max / mean / std = 1 / 1 / 1 / 0
  per-element range min / mean / max = 0 / 0 / 0
  first 8 per-element ranges: 0, 0, 0, 0, 0, 0, 0, 0

Field: angle
  size = 4800 × 4
  global min / max / mean / std = 0 / 0 / 0 / 0
  per-element range min / mean / max = 0 / 0 / 0
  first 8 per-element ranges: 0, 0, 0, 0, 0, 0, 0, 0

Field: μ_l
  size = 4800 × 4
  global min / max / mean / std = 2.45273 / 25.3546 / 9.76921 / 4.31268
  per-element range min / mean / max = 0.0100141 / 1.313 / 4.66301
  first 8 per-element ranges: 0.399133, 0.553196, 0.626925, 0.630988, 0.601763, 0.642586, 0.617257, 0.522188

Field: μ_t
  size = 4800 × 4
  global min / max / mean / std = 0.599931 / 1.694

In [20]:
mf = build_material_field(fields; use_centroids=false, eltype_out=Float32)
mat = get_material(mf, 10, 3) 

(μ_l = 8.830199f0, μ_t = 0.99992853f0, α = 5.4199247f0, β = 1.9061143f0, λ = 1.0f0, angle = 0.0f0)